# Louisiana 64-County Model Comparison

This notebook compares the Neighbor Network, identity-fixed No
County Network, and NB-AR using released test summaries only.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

ROOT = find_repo_root()
TABLE_DIR = ROOT / "results" / "tables"
MATRIX_DIR = ROOT / "results" / "matrices"

In [ ]:
results = pd.read_csv(TABLE_DIR / "table_6_all64_three_model_comparison.csv")
results

In [ ]:
order = ["Neighbor Network", "No County Network", "NB-AR"]
results = results.set_index("model").loc[order].reset_index()
specs = [("test_nll", "test_nll_sd", "Test NLL"),
         ("mae", "mae_sd", "MAE"),
         ("mse", "mse_sd", "MSE")]
colors = ["#177E89", "#73777F", "#D97706"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for axis, (metric, error, label) in zip(axes, specs):
    axis.bar(results["model"], results[metric], yerr=results[error],
             color=colors, capsize=4)
    axis.set_ylabel(label)
    axis.tick_params(axis="x", rotation=20)
    axis.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

In [ ]:
indexed = results.set_index("model")
neighbor = indexed.loc["Neighbor Network"]
no_network = indexed.loc["No County Network"]
nb_ar = indexed.loc["NB-AR"]
pd.Series({
    "MSE reduction vs No County Network (%)": 100 * (1 - neighbor.mse / no_network.mse),
    "MAE reduction vs No County Network (%)": 100 * (1 - neighbor.mae / no_network.mae),
    "MSE reduction vs NB-AR (%)": 100 * (1 - neighbor.mse / nb_ar.mse),
    "MAE reduction vs NB-AR (%)": 100 * (1 - neighbor.mae / nb_ar.mae),
}).round(2)